# 17. 리더보드 대리 평가(proxy) 만들기
제출 1회 남음 → 로컬에서 리더보드 점수를 예측할 기준이 필요.
LB 점수를 아는 9/1 제출 파일들을, 급등 국면 최신 데이터로 만든 **가짜 정답**으로 채점해 실제 LB 순서와 맞는지 확인.

In [1]:
import os, duckdb, numpy as np, pandas as pd
from scipy.stats import spearmanr, rankdata
from sklearn.metrics import average_precision_score
RAW, SUBD = '../data/raw/epoch_data', '../submissions'
con = duckdb.connect()
con.sql(f"CREATE VIEW v AS SELECT * FROM read_csv('{RAW}/video_5d_views.csv', header=true, auto_detect=true)")
sample = pd.read_csv(f'{RAW}/sample_submission.csv', encoding='utf-8-sig')
sample['channel_id'] = sample.row_id.str.rsplit('_', n=1).str[0]
LB = {'submit_04_logreg_rankblend_fut26.csv': 0.4554, 'submit_05_logreg_q75rule_a08.csv': 0.4402,
      'submit_15_M4fix_shockadj_previnv.csv': 0.6110, 'submit_17_M4fix_pastdeflate.csv': 0.5990,
      'submit_21_ens_s4_s3_s15.csv': 0.6420, 'submit_36_ens21_prevrule30.csv': 0.6696,
      'submit_38_ens21_prevrule50.csv': 0.6707, 'submit_42_rule50_withinV1.csv': 0.6483,
      'submit_10_s2_cons_defl.csv': 0.6150, 'submit_33_ens21_adj.csv': 0.6357, 'submit_35_ens21_persist30.csv': 0.6380}
LB = {k: v for k, v in LB.items() if os.path.exists(f'{SUBD}/{k}')}
P = pd.DataFrame({k: pd.read_csv(f'{SUBD}/{k}').set_index('row_id').prediction.loc[sample.row_id].values for k in LB}, index=sample.channel_id.values)
print(len(LB), '개 파일')

def pseudo(past_end, fut_start, fut_end, q=0.8):
    g = con.sql(f"""
      SELECT channel_id,
             LN(1 + MEDIAN(view_5d) FILTER (WHERE published_at < TIMESTAMP '{past_end}')) AS lp,
             LN(1 + COALESCE(MEDIAN(view_5d) FILTER (WHERE published_at >= TIMESTAMP '{fut_start}' AND published_at < TIMESTAMP '{fut_end}'), 0)) AS lf,
             COUNT(*) FILTER (WHERE published_at >= TIMESTAMP '{fut_start}' AND published_at < TIMESTAMP '{fut_end}') AS nf
      FROM v WHERE is_shorts = 0 AND view_5d IS NOT NULL GROUP BY 1""").df().set_index('channel_id')
    g = g.reindex(P.index)
    gr = g.lf - g.lp
    return (gr >= gr.quantile(q)).astype(int), g.nf

proxies = {
    'A: 과거<08-25, 미래 08-25~09-01': ('2026-08-25', '2026-08-25', '2026-09-01'),
    'B: 과거<08-27, 미래 08-27~09-01': ('2026-08-27', '2026-08-27', '2026-09-01'),
    'C: 과거<08-29, 미래 08-29~09-01': ('2026-08-29', '2026-08-29', '2026-09-01'),
    'D: 과거<08-22, 미래 08-22~09-01': ('2026-08-22', '2026-08-22', '2026-09-01'),
}
rows = {}
for name, (pe, fs, fe) in proxies.items():
    y, nf = pseudo(pe, fs, fe)
    sc = {k: average_precision_score(y, P[k]) for k in LB}
    rho = spearmanr(list(sc.values()), list(LB.values()))[0]
    rows[name] = {'LB와 순위상관': round(rho, 3), **{k[:22]: round(v, 3) for k, v in sc.items()}}
R = pd.DataFrame(rows).T
print(R[['LB와 순위상관']])
print(pd.DataFrame(rows).iloc[1:].assign(LB=[LB[k] for k in LB]).to_string())

11 개 파일


                             LB와 순위상관
A: 과거<08-25, 미래 08-25~09-01     0.682
B: 과거<08-27, 미래 08-27~09-01     0.627
C: 과거<08-29, 미래 08-29~09-01     0.627
D: 과거<08-22, 미래 08-22~09-01     0.273
                        A: 과거<08-25, 미래 08-25~09-01  B: 과거<08-27, 미래 08-27~09-01  C: 과거<08-29, 미래 08-29~09-01  D: 과거<08-22, 미래 08-22~09-01      LB
submit_04_logreg_rankb                        0.465                        0.501                        0.417                        0.449  0.4554
submit_05_logreg_q75ru                        0.535                        0.529                        0.426                        0.529  0.4402
submit_15_M4fix_shocka                        0.499                        0.502                        0.410                        0.470  0.6110
submit_17_M4fix_pastde                        0.490                        0.482                        0.386                        0.460  0.5990
submit_21_ens_s4_s3_s1                        0.543                        

### 결과: proxy–LB 순위상관 0.63~0.68 (중간)
- 최근 영상에 기대는 파일(05, 35)을 과대평가 (가짜 정답이 08-25~31 영상으로 만들어짐)
- **같은 계열(submit_21 + 8/25 규칙) 안에서는 LB 순서와 일치**: 21 < 36 < 38, 42 < 38
- 8/25 라벨 규칙도 과대평가할 수 있음 (36→38: LB +0.001, proxy A +0.007)
→ 같은 계열 안의 후보 선택에만 사용

## 마지막 1회 후보: submit_21 + 8/25 규칙 계열 변형

In [2]:
tl = pd.read_csv(f'{RAW}/train_labels.csv'); tl['channel_id'] = tl.row_id.str.rsplit('_', n=1).str[0]
prev = tl[tl.row_id.str.endswith('2026-08-25')].set_index('channel_id').target.reindex(P.index)
lp = con.sql("""SELECT channel_id, LN(1 + MEDIAN(view_5d)) lp FROM v WHERE is_shorts = 0 AND view_5d IS NOT NULL
                AND published_at < TIMESTAMP '2026-09-01' GROUP BY 1""").df().set_index('channel_id').lp.reindex(P.index)
N = len(P); r = lambda x: rankdata(pd.Series(np.asarray(x, float)).fillna(np.nanmedian(x))) / N
r21 = r(P['submit_21_ens_s4_s3_s15.csv']); prev_f = prev.fillna(0.2).values
def rule(alpha):   # alpha: 그룹 안에서 과거중앙값 낮은 순의 영향력 (팀원 규칙은 1e-3)
    return r(prev_f + alpha * r(-lp))
C = {}
for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    for a in [1e-3, 0.1, 0.3]:
        C[f'w={w}, a={a}'] = (1 - w) * r21 + w * rule(a)
C['gate(40)'] = prev_f + 0.5 * r21
C['gate + 저과거 0.3'] = prev_f + 0.5 * (0.7 * r21 + 0.3 * r(-lp))
C['submit_43'] = pd.read_csv(f'{SUBD}/submit_43_prevgate_withinV1.csv').set_index('row_id').prediction.loc[sample.row_id].values
ys = {n: pseudo(*a)[0] for n, a in proxies.items() if not n.startswith('D')}
tab = pd.DataFrame({n: {k: average_precision_score(y, v) for k, v in C.items()} for n, y in ys.items()})
tab['평균(A,B,C)'] = tab.mean(axis=1)
tab['상위139 라벨=1'] = [int((prev.values[np.argsort(-np.asarray(v))[:139]] == 1).sum()) for v in C.values()]
ref = tab.loc['w=0.5, a=0.001', '평균(A,B,C)']
tab['38 대비'] = tab['평균(A,B,C)'] - ref
print(tab.round(4).sort_values('평균(A,B,C)', ascending=False).to_string())

                A: 과거<08-25, 미래 08-25~09-01  B: 과거<08-27, 미래 08-27~09-01  C: 과거<08-29, 미래 08-29~09-01  평균(A,B,C)  상위139 라벨=1   38 대비
w=0.7, a=0.3                         0.5728                       0.6062                       0.4986     0.5592         130  0.0125
w=0.7, a=0.1                         0.5722                       0.6054                       0.4980     0.5585         130  0.0118
w=0.7, a=0.001                       0.5722                       0.6054                       0.4980     0.5585         130  0.0118
gate + 저과거 0.3                       0.5714                       0.6001                       0.4952     0.5556         134  0.0089
w=0.6, a=0.3                         0.5667                       0.5981                       0.4952     0.5534         128  0.0067
w=0.6, a=0.001                       0.5668                       0.5980                       0.4948     0.5532         128  0.0065
w=0.6, a=0.1                         0.5668                       0.5

### 후보 비교 결과: 규칙 0.7이 proxy 1위(+0.012)지만, proxy는 규칙 비중을 과대평가(36→38: proxy +0.011, LB +0.001) → LB 약 +0.001 예상.

## 0.7 도전: 급등 국면을 반영한 그룹 안 순서
가설: 42가 떨어진 이유 = V1은 급등 전 데이터로 학습해 "K(08-25~31)가 높음"을 좋은 신호로 배움.
본선에선 K가 부풀어 있고 정답의 과거 중앙값에 K가 포함 → **K가 높아 라벨=1이 된 채널은 오히려 불리**, **K가 낮은데 라벨=1인 채널은 9월 초 영상(U)이 높았다는 뜻 → 매우 유리**.
이 구분을 급등 수준(ŝ=1.0, t̂=1.25)·원값 과거로 직접 계산한 점수 = 조건부 시뮬 `sim_cond` (submit_15 파이프라인).
먼저 sim_cond가 이 구분대로 움직이는지 확인.

In [3]:
import pickle
dd = pickle.load(open('../data/processed/_refine_cache.pkl', 'rb'))
T = dd['test'].set_index('channel_id').reindex(P.index)
T['kn'] = T.K.apply(lambda x: len(x) if isinstance(x, (list, np.ndarray)) else 0)
T['kmed_raw'] = T.K.apply(lambda x: np.median(x) if isinstance(x, (list, np.ndarray)) and len(x) else np.nan)
T['k_vs_prevpast'] = T.kmed_raw - T.past_l_prev          # K가 8/25 과거 대비 얼마나 높았나 (원값)
T['r21'] = r21; T['r38'] = r(P['submit_38_ens21_prevrule50.csv'])
g1 = T[T.prev == 1].copy()
g1['K수준'] = pd.qcut(g1.k_vs_prevpast.rank(method='first'), 3, labels=['K 낮음', 'K 중간', 'K 높음'])
print('라벨=1 그룹 (134채널): K 수준별 평균')
print(g1.groupby('K수준', observed=True)[['sim_cond', 'r21', 'r38', 'kn', 'past_l']].mean().round(3).assign(n=g1.groupby('K수준', observed=True).size()))
print()
print('라벨=1 그룹 내 순위상관: sim_cond vs r21', round(spearmanr(g1.sim_cond, g1.r21)[0], 3),
      '| sim_cond vs K수준', round(spearmanr(g1.sim_cond, g1.k_vs_prevpast)[0], 3),
      '| r21 vs K수준', round(spearmanr(g1.r21, g1.k_vs_prevpast)[0], 3))

라벨=1 그룹 (134채널): K 수준별 평균
      sim_cond    r21    r38     kn  past_l   n
K수준                                            
K 낮음     0.421  0.906  0.910  4.190   7.835  42
K 중간     0.479  0.865  0.887  4.732   8.209  41
K 높음     0.543  0.869  0.895  5.024   8.017  42

라벨=1 그룹 내 순위상관: sim_cond vs r21 0.384 | sim_cond vs K수준 nan | r21 vs K수준 nan


### 결과: sim_cond는 K 높은 채널을 선호(가설과 반대 — 9월 영상을 최근 분포에서 뽑기 때문). 반면 **LB가 좋은 submit_21·38은 K 낮은 라벨=1 채널을 가장 위에** 둠.
### 확인: LB가 나빴던 42(V1)는 K 높은 쪽을 선호했나? 각 제출의 라벨=1 그룹 내 순서 vs K 수준

In [4]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
po = dd['po']; FC, INTER = dd['FC'], dd['INTER']
lr_ = Pipeline([('imp', SimpleImputer(strategy='median')), ('sc', StandardScaler()), ('m', LogisticRegression(max_iter=3000, C=0.01, class_weight='balanced'))])
T['v1'] = lr_.fit(po[FC + INTER], po.target).predict_proba(dd['test'].set_index('channel_id').reindex(P.index)[FC + INTER])[:, 1]
T['r42'] = r(P['submit_42_rule50_withinV1.csv']); T['r36'] = r(P['submit_36_ens21_prevrule30.csv'])
T['r15'] = r(P['submit_15_M4fix_shockadj_previnv.csv'])
for f, lbv in [('submit_10_s2_cons_defl.csv', 0.615)]:
    T['r_s2'] = r(P[f])
g1 = T[(T.prev == 1) & T.k_vs_prevpast.notna()]
lbmap = {'r21': 0.642, 'r36': 0.6696, 'r38': 0.6707, 'r42': 0.6483, 'r15': 0.611, 'r_s2': 0.615, 'v1': None, 'sim_cond': None}
rows = []
for c, lbv in lbmap.items():
    rows.append({'점수': c, 'LB': lbv, '라벨=1 그룹 내 K수준과 순위상관': round(spearmanr(g1[c], g1.k_vs_prevpast)[0], 3)})
print(pd.DataFrame(rows).to_string(index=False))
g0 = T[(T.prev == 0) & T.k_vs_prevpast.notna()]
print('참고 — 라벨=0 그룹 내 K수준과 순위상관:', {c: round(spearmanr(g0[c], g0.k_vs_prevpast)[0], 3) for c in ['r21', 'r38', 'r42', 'v1', 'sim_cond']})

      점수     LB  라벨=1 그룹 내 K수준과 순위상관
     r21 0.6420               -0.097
     r36 0.6696               -0.086
     r38 0.6707               -0.083
     r42 0.6483               -0.144
     r15 0.6110                0.001
    r_s2 0.6150               -0.263
      v1    NaN               -0.143
sim_cond    NaN                0.134
참고 — 라벨=0 그룹 내 K수준과 순위상관: {'r21': np.float64(0.307), 'r38': np.float64(0.247), 'r42': np.float64(0.401), 'v1': np.float64(0.483), 'sim_cond': np.float64(0.172)}


### 결과
- 라벨=1 그룹: 모든 점수가 K와 약한 음의 상관(−0.08~−0.14) → 가설로 42 하락 설명 불가
- **라벨=0 그룹**: K와의 상관 38 0.247 (LB 0.6707) < 21 0.307 (0.642) < 42 0.401 (0.6483). "K 높은데 라벨=0" = 9월 초 영상이 낮았음 → 본선 불리. 42는 이들을 가장 강하게 올렸고 하락.

## submit_44: 38 + 라벨=0 그룹 안 순서만 "K 낮은 채널 우선"으로 (라벨=1 그룹은 38 그대로)

In [5]:
t38 = T.r38.values.copy()
lowK = r(-T.k_vs_prevpast.fillna(T.k_vs_prevpast.median()))
is0 = (T.prev == 0).values
s44 = t38.copy()
# 라벨=0 그룹 안: 38 점수의 그룹 내 범위는 유지하고 순서만 0.5·38 + 0.5·K낮음으로 재배열
g = np.where(is0)[0]
new_order = rankdata(0.5 * rankdata(t38[g]) + 0.5 * rankdata(lowK[g]))
s44[g] = np.sort(t38[g])[(new_order - 1).astype(int)]
s44 = s44 + 1e-9 * rankdata(t38)
top38, top44 = set(np.argsort(-t38)[:139]), set(np.argsort(-s44)[:139])
print('상위139 중 바뀐 채널 수:', len(top38 - top44))
out_ = list(top38 - top44); in_ = list(top44 - top38)
print('빠진 채널(라벨=0) K수준 평균', round(np.nanmean(T.k_vs_prevpast.values[out_]), 3), '| 들어온 채널 K수준 평균', round(np.nanmean(T.k_vs_prevpast.values[in_]), 3))
print('38과 전체 순위상관', round(spearmanr(t38, s44)[0], 4))
sv = rankdata(s44) / N; sv = (sv - sv.min()) / (sv.max() - sv.min())
sub = pd.DataFrame({'row_id': sample.row_id.values, 'prediction': sv})
assert len(sub) == 694 and sub.row_id.is_unique and sub.prediction.between(0, 1).all() and not sub.prediction.duplicated().any()
sub.to_csv(f'{SUBD}/submit_44_ens38_label0_lowK.csv', index=False, encoding='utf-8')
ys = {n: pseudo(*a)[0] for n, a in proxies.items() if not n.startswith('D')}
print('proxy 평균 (참고, K 편향 있음): 38', round(np.mean([average_precision_score(y, t38) for y in ys.values()]), 4),
      '| 44', round(np.mean([average_precision_score(y, s44) for y in ys.values()]), 4))

상위139 중 바뀐 채널 수: 5
빠진 채널(라벨=0) K수준 평균 0.758 | 들어온 채널 K수준 평균 -0.852
38과 전체 순위상관 0.8164


proxy 평균 (참고, K 편향 있음): 38 0.547 | 44 0.5029
